<a href="https://colab.research.google.com/github/ayd7n/Gorunmez_Balik/blob/main/Block_40.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Öncelikle python binance kütüphanemizi kuralım.
!pip install python-binance

In [ ]:
#@title Gerekli İmportlarımızı Yapalım
from binance.client import Client
from binance.helpers import round_step_size
import pandas as pd
import datetime
from datetime import datetime
import sqlite3
import pandas as pd
import numpy as np
import time
import json
import ast
import math
import os


from IPython.core.interactiveshell import InteractiveShell
from statistics import mean

from __future__ import print_function
from ipywidgets import interact, interactive, fixed, interact_manual, HTML
import ipywidgets as widgets

# Bunu yapma amacımız sayıları e şeklinde göstermesin diye.
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
#@title Api Bilgilerimizi Girelim.
api_key = "WKOT7m0nL7KSHg7Lu5ZaL9dVvzQ3RaCbBtQICkSZGrmrJP4i8zUhiNdzJtI0i1Xb"
secret = "iZuRLwyGqkSqxYNva2UflVcf15rakpkaNNxyrxgI3pL0LxIeItm4w33MK8q3kmfE"

client = Client(api_key, secret, testnet=True)

print(client.ping())
print("Baglanti_Basarili")

{}
Baglanti_Basarili


In [ ]:
#@title IO Genel Modülü
from google.colab import drive
drive.mount('/content/drive')
veritabani = "/content/drive/MyDrive/Veritabani/level.db"

def Tek_Hucre(komut):
    komut = komut.strip()
    try:
        conn = sqlite3.connect(veritabani)
        c = conn.cursor()
        c.execute(komut)
        veri = c.fetchone()
        conn.close()
        return str(veri[0])
    except:
        return "Hata"


def DuzSorgu(Sorgu):
    Sorgu = Sorgu.strip()
    # Bu method ile kolon başlıkları ile birlikte tek satırlık veri çekeriz.
    conn = sqlite3.connect(veritabani)
    conn.row_factory = sqlite3.Row
    c = conn.cursor()
    oku = c.execute(Sorgu)
    veri = oku.fetchone()
    conn.close()
    return veri


def PandasSQL(Sorgu):
    Sorgu = Sorgu.strip()
    # Doğrudan dataframe olarak tabloyu alırız.
    conn = sqlite3.connect(veritabani)
    tablo = pd.read_sql_query(Sorgu, conn)
    return tablo


def PandasYaz(dbdeki_alan, df):
    # Pandas kullanarak tabloyu yazarız.
    conn = sqlite3.connect(veritabani)
    df.to_sql(dbdeki_alan, conn, if_exists="append", index=False)


def Pandas_Silip_Yaz(dbdeki_alan, df):
    # Pandas kullanarak tabloyu yazarız.
    conn = sqlite3.connect(veritabani)
    df.to_sql(dbdeki_alan, conn, if_exists="replace", index=False)


def Komut(Sorgu):
    Sorgu = Sorgu.strip()
    try:
        conn = sqlite3.connect(veritabani)
        c = conn.cursor()
        c.execute(Sorgu)
        conn.commit()
        conn.close()
        return "Basarili"
    except Exception as e:
        return str(e)


def Json_PandasSQL(Sorgu):
    Sorgu = Sorgu.strip()
    conn = sqlite3.connect(veritabani)
    tablo = pd.read_sql_query(Sorgu, conn).to_json(orient="records")
    return tablo

Mounted at /content/drive


In [ ]:
#@title DB Viewer

#@title Arayüz Fonksiyonları
def Df_Stilcisi(df , yuvarlanacak_sayi = 8):

    if isinstance(df, pd.Series):
        df = pd.DataFrame(df , columns = ["Değerler"])

    df = df.round(yuvarlanacak_sayi)
    baslik_ozellikleri = [('font-size', '12px'),('background-color', '#579BB1'),('color','#03001C'),("border", "1px solid black")]
    hucre_ozellikleri = [('font-size', '12px'), ("text-align" , "left"),("max-width" , "600px")]
    index_ozellikleri = [('text-align', 'left'),('font-size', '12px'),('color','#ECE8DD')]

    dfstyle = [dict(selector="thead", props=baslik_ozellikleri), dict(selector="td", props=hucre_ozellikleri),dict(selector="tbody", props= [("border", "1px solid black")]) ,dict(selector="tbody th", props=index_ozellikleri),]
    df = df.style.set_table_styles(dfstyle)

    return df

def Stilli_Bas(df):
    display(Df_Stilcisi(df))
    
display(HTML("<style>.yukseklik { height: 100% ; width: 2000px; overflow: visible}</style>"))

output = widgets.Output().add_class("yukseklik")

tablo_listesi = PandasSQL('SELECT tbl_name FROM sqlite_master where type IN ("table" , "view")')
tablo_listesi = tablo_listesi['tbl_name'].tolist()


tablolar = widgets.Dropdown(
    options=tablo_listesi,
    description='Tabloyu Seçiniz:',
    disabled=False,
)

query = widgets.Textarea(
    value='',
    description='Sorgu:',
    disabled=False
)

# verileri_goster butonunun alanı
verileri_goster = widgets.Button(description="Verileri Goster")

def goster(b):
    with output:
        output.clear_output()

        df = pd.DataFrame()

        if query.value == '':
            df = PandasSQL(f'select * from {tablolar.value}')
        else:
            df = PandasSQL(query.value)

        display(Df_Stilcisi(df))



verileri_goster.on_click(goster)
# verileri_goster butonunun alanı

yatay_butonlar = widgets.HBox([tablolar,verileri_goster])
widgets.VBox([yatay_butonlar ,query, output])

HTML(value='<style>.yukseklik { height: 100% ; width: 2000px; overflow: visible}</style>')

In [ ]:
#@title Yardımcı Fonksiyonlar

def Tarih_Normallestirici(tarih):
    timestamp = int(tarih) / 1000 # saniye cinsinden zaman damgası
    dt_object = datetime.fromtimestamp(timestamp)
    formatted_datetime = dt_object.strftime("%Y-%m-%d %H:%M:%S")
    return formatted_datetime

def Tum_Sembolleri_Al():
    exchange_info = client.get_exchange_info()
    semboller = pd.DataFrame(exchange_info['symbols']).applymap(str)
    Pandas_Silip_Yaz("exchange_info", semboller)

def Tum_Fiyatlari_Al():
    tickers = client.get_all_tickers()
    tickers = pd.DataFrame(tickers)
    Pandas_Silip_Yaz("tickers",tickers)

def Mevcut_Fiyati_Ver(symbol):
    # Borsadan alınmış son fiyata bakalım.
    tickers = PandasSQL("Select * from tickers")
    mevcut_fiyat = tickers[tickers["symbol"] == symbol]["price"].values[0]
    mevcut_fiyat = float(mevcut_fiyat)

    return mevcut_fiyat
    # Borsadan alınmış son fiyata bakalım.

def Satis_Plani_Olusturucu(k_id):
    '''
    Bu fonksiyondan istediklerimiz.
    1-Künyedeki toplam p1 miktarını kullanacağız.
    2-Künyedeki parametreleri kullanacağız.
    3-Bu doğrultuda yapılan bir satış planı emir yollama kriterlerine uygun mu
    '''

    kunye = PandasSQL(f"SELECT * FROM kunye WHERE k_id = {k_id};").loc[0]

    spamyo = kunye["satis_parcalar_arasi_yuzde_orani"] # Satış parçaları arası min yüzde oranı
    ilk_satis_sicrama_yuzdesi = kunye["ilk_satis_sicrama_yuzdesi"]
    parca_sayisi = kunye["parca_sayisi"]
    p1_miktari = kunye["p1_miktari"]
    sembol = kunye["sembol"]

    gelen_fiyat = Mevcut_Fiyati_Ver(sembol)

    ilk_parcanin_satis_fiyati = gelen_fiyat*(100+ilk_satis_sicrama_yuzdesi)/100

    satis_fiyatlari = [ilk_parcanin_satis_fiyati]
    for i in range(parca_sayisi-1):
        listeye_eklenecek_eleman = max(satis_fiyatlari)*(100+spamyo)/100
        satis_fiyatlari.append(listeye_eklenecek_eleman)

    satilacak_birim_parca_miktari = p1_miktari/parca_sayisi

    satis_plani = pd.DataFrame()
    satis_plani["fiyat"] = satis_fiyatlari
    satis_plani["adet"] = satilacak_birim_parca_miktari

    # Şimdi bu fiyat ve bu adetlerde emir yollayabilir miyiz bunu kontrol edelim.
    sonuc = satis_plani.apply(lambda x: Limit_Emri_Oncesi_Kontrolcusu(sembol, x["adet"] , x["fiyat"]) , axis = 1)
    satis_plani[["sonuc","duzenlenmis_price","duzenlenmis_quantity", "aciklama"]] = pd.DataFrame(sonuc.tolist())

    durum = False
    durum = satis_plani["sonuc"].all()
    if durum:
        satis_plani = satis_plani[["duzenlenmis_price","duzenlenmis_quantity"]]
        satis_plani.columns = ["fiyat" , "adet"]
      
    return durum , satis_plani

In [ ]:
#@title Emirlerle İlgili Yardımcı Fonksiyonlar

def DBden_Emir_Sil(k_id, symbol, orderId, tablo_ismi):
    sorgu_metni = f'''
    DELETE FROM {tablo_ismi}
      WHERE k_id = '{k_id}' AND 
            symbol = '{symbol}' AND 
            orderId = '{orderId}'
    '''
    Komut(sorgu_metni)

def Order_Blokaj_Etkisi(k_id, order):
    fiyat = float(order["price"])
    emir_miktari = float(order["origQty"])
    alis_veya_satis = order["side"]

    blokeli_p1e_etki = 0.0
    blokeli_p2ye_etki = 0.0

    if alis_veya_satis == "SELL":
        blokeli_p1e_etki = -emir_miktari
    else:
        blokeli_p2ye_etki = -emir_miktari * fiyat

    sorgu_metni = f'''
    UPDATE kunye
    SET blokeli_p1_miktari = blokeli_p1_miktari + {blokeli_p1e_etki},
        blokeli_p2_miktari = blokeli_p2_miktari + {blokeli_p2ye_etki}
    WHERE k_id = {k_id};
    '''
    Komut(sorgu_metni)

def Trade_Kasa_Etkisi(k_id, symbol, orderId):
    tradeler = pd.DataFrame(client.get_my_trades(symbol=symbol, orderId=orderId))

    # Toplamları alalım.
    toplam_qty = tradeler["qty"].astype(float).sum()  #Paritenin birinci elemanı
    toplam_quoteQty = tradeler["quoteQty"].astype(float).sum()  #Paritenin ikinci elemanı
    toplam_commission = tradeler["commission"].astype(float).sum()  #Komisyon
    # Toplamları alalım.

    p1e_etki = 0.0
    p2ye_etki = 0.0

    alici_miyiz = tradeler.loc[0]["isBuyer"]

    if alici_miyiz:
        p1e_etki = toplam_qty - toplam_commission  # Ele geçecek miktardan komisyon düşülür.
        p2ye_etki = -toplam_quoteQty  # Cepten quoteQty toplamı kadar para çıkar.
    else:
        # Eğer satıcıysak.
        p1e_etki = - toplam_qty  # Cepten qty toplamı kadar p1 çıkar.
        p2ye_etki = toplam_quoteQty - toplam_commission  # Cebe quoteQty - komisyon kadar para girer.

    # Veritabanından da değerlerimizi güncelleyelim.
    sorgu_metni = f'''
    UPDATE kunye
    SET p1_miktari = p1_miktari + {p1e_etki},
        p2_miktari = p2_miktari + {p2ye_etki}
    WHERE k_id = {k_id};
                                             
    '''
    return Komut(sorgu_metni)

def Emir_Yok_Edici(k_id, symbol, orderId):
    # Siparişin son durumunu çekelim.
    order = client.get_order(symbol=symbol, orderId=orderId)

    durum = order["status"]
    executedQty = float(order["executedQty"])

    # Eğer emir iptal edilmiş , reddedilmiş , süresi geçmiş veya gerçekleşmiş ise ve emrin bir kısmı gerçekleşmiş ise
    # o zaman kasaya etki etmiş demektir ve kasada değişiklik de yapar.
    if durum in ['CANCELED', 'REJECTED', 'EXPIRED', 'FILLED']:
        if executedQty > 0:
            Trade_Kasa_Etkisi(k_id, symbol, orderId)

        # Eğer executedQty sıfırdan küçükse kasada etkisi olmadığı için blokaj kaldırılıp , emir silinir.
        Order_Blokaj_Etkisi(k_id, order)
        DBden_Emir_Sil(k_id, symbol, orderId, "kapanmamis_emirler")
        DBden_Emir_Sil(k_id, symbol, orderId, "alis_plani")
        DBden_Emir_Sil(k_id, symbol, orderId, "satis_plani")
        #return f"Success* Emir zaten {durum} statusunde olduğu için db operasyonlari yapildi. executedQty = {executedQty}"
        return True, durum, executedQty

    # Emir açık veya kısmi gerçekleşmiş de olabilir. Teorik olarak emir açıktır.
    # Emir iptali sadece bu aşamada gönderilir.
    elif durum in ["NEW", "PARTIALLY_FILLED"]:
        iptal_talebi = {}
        try:
            # Emir iptal talebinde bulunalım.
            iptal_talebi = client.cancel_order(symbol=symbol, orderId=orderId)
        except:
            return False, durum, executedQty
        if iptal_talebi["status"] == "CANCELED":  # Emir başarıyla gerçekleştiyse.

            iptal_edilen_emrin_executedQty_miktari = float(iptal_talebi["executedQty"])
            if iptal_edilen_emrin_executedQty_miktari > 0:
                Trade_Kasa_Etkisi(k_id, symbol, orderId)

            Order_Blokaj_Etkisi(k_id, order)

            DBden_Emir_Sil(k_id, symbol, orderId, "kapanmamis_emirler")
            DBden_Emir_Sil(k_id, symbol, orderId, "alis_plani")
            DBden_Emir_Sil(k_id, symbol, orderId, "satis_plani")
            return True, durum, executedQty
        else:
            False, durum, executedQty
    elif durum == 'PENDING_CANCEL':
        # Buraya wait gibi bir kod da koyabiliriz. Çünkü iptal bekleniyordur.
        return False, durum, executedQty

def Kunyedeki_Emirleri_Temizle(k_id):
    # Önce dbdeki bu künyeye ait açık emirleri çekelim.
    sorgu_metni = f'''
    SELECT *
    FROM kapanmamis_emirler
    WHERE k_id = {k_id};
    '''
    kapanmamis_emirler = PandasSQL(sorgu_metni)

    if len(kapanmamis_emirler.index) == 0:
        return True , kapanmamis_emirler

    # Eğer emir kapanmadıysa 3 kere tekrar yollar.
    def Emirleri_Kapaticiya_Yolla(x):

        iptal_edildi_mi = False
        max_tekrar = 3
        tekrar = 0

        while not iptal_edildi_mi and tekrar < max_tekrar:
            iptal_edildi_mi, emrin_son_durumu, executedQty = Emir_Yok_Edici(x["k_id"], x["symbol"], x["orderId"])
            tekrar += 1

            # Eğer iptal edilmediyse sonraki tekrara geçmeden 3 saniye beklesin.
            if not iptal_edildi_mi:
                time.sleep(3)

        if iptal_edildi_mi:
            return True
        else:
            return False

    kapanmamis_emirler["kapanma_durumu"] = kapanmamis_emirler.apply(lambda x: Emirleri_Kapaticiya_Yolla(x), axis=1)

    if kapanmamis_emirler["kapanma_durumu"].all():
        return True , kapanmamis_emirler
    else:
        return False ,kapanmamis_emirler

def Limit_Emri_Oncesi_Kontrolcusu(symbol, quantity , price):
    '''
    **Limit Emri Öncesi Kontrolcüsü**
    Limit emri yollamadan 3 farklı kriterde emri kontrol edelim.

    **Kontroller**

    **1 PRICE_FILTER**
    - price >= minPrice
    - price <= maxPrice
    - price % tickSize == 0

    **2 MIN NOTIONAL**

    MIN NOTIONAL kuralına göre fiyat*miktar > MIN NOTIONAL olmalıdır.

    **3 LOT SIZE**
    LOT SIZE kontrolü
    - quantity >= minQty
    - quantity <= maxQty
    - (quantity-minQty) % stepSize == 0
    '''


    sorgu_metni = f'''
    SELECT *
    FROM exchange_info
    WHERE symbol = '{symbol}';
    '''
    
    kurallar = PandasSQL(sorgu_metni).head(1)
    filtreler = kurallar["filters"].values[0]

    list_data = ast.literal_eval(filtreler)
    fixed_data = json.dumps(list_data)
    dict_data = json.loads(fixed_data)

    # PRICE_FILTER kontrolü
    PRICE_FILTER_items = [item for item in dict_data if item['filterType'] == 'PRICE_FILTER']
    minPrice = float(PRICE_FILTER_items[0]['minPrice'])
    maxPrice = float(PRICE_FILTER_items[0]['maxPrice'])
    tickSize = float(PRICE_FILTER_items[0]['tickSize'])

    # minNotional kontrolü
    minNotional_items = [item for item in dict_data if item['filterType'] == 'MIN_NOTIONAL']
    minNotional = float(minNotional_items[0]['minNotional'])

    # Qty kontrolü
    miktar_filtreleri = [item for item in dict_data if item['filterType'] == 'LOT_SIZE']
    minQty = float(miktar_filtreleri[0]['minQty'])
    maxQty = float(miktar_filtreleri[0]['maxQty'])
    stepSize = float(miktar_filtreleri[0]['stepSize'])

    ## fiyat ve miktarı yuvarlayalım ###
    duzenlenmis_price = round_step_size(price, tickSize)
    duzenlenmis_quantity = round_step_size(quantity, stepSize)

    if duzenlenmis_price < minPrice or duzenlenmis_price > maxPrice:
        return False ,duzenlenmis_price,duzenlenmis_quantity , f"PRICE_FILTER filtresine takıldı. tickSize = {tickSize} , istenen fiyat = {price} , minPrice = {minPrice} , maxPrice = {maxPrice}"
    
    if duzenlenmis_quantity < minQty or duzenlenmis_quantity > maxQty:
        return False ,duzenlenmis_price,duzenlenmis_quantity , f"LOT_SIZE filtresine takıldı. Stepsize = {stepSize} ,istenen Miktar = {quantity} minQty = {minQty} , maxQty = {maxQty}"
    
    if duzenlenmis_price*duzenlenmis_quantity <= minNotional:
        return False ,duzenlenmis_price,duzenlenmis_quantity , f"minNotional filtresine takıldı. minNotional = {minNotional} ,price*quantity = {duzenlenmis_price*duzenlenmis_quantity} "

    return True ,duzenlenmis_price,duzenlenmis_quantity ,"sorun_yok"

def Kapanmamis_Emirlere_Kayit_Ekle(k_id,order):
    workingTime = Tarih_Normallestirici(order["workingTime"])
    symbol = order["symbol"]
    orderId = int(order["orderId"])
    price = order["price"]
    origQty = order["origQty"]
    type = order["type"]
    side = order["side"]

    sorgu_metni = f'''
    INSERT INTO kapanmamis_emirler (
                                   k_id,
                                   symbol,
                                   orderId,
                                   price,
                                   origQty,
                                   workingTime,
                                   type,
                                   side
                               )
                               VALUES (
                                   '{k_id}',
                                   '{symbol}',
                                   '{orderId}',
                                   '{price}',
                                   '{origQty}',
                                   '{workingTime}',
                                   '{type}',
                                   '{side}'
                               );
    '''
    return Komut(sorgu_metni)

def Satis_Planina_Kayit_Ekle(k_id,order):
    symbol = order["symbol"]
    orderId = int(order["orderId"])
    fiyat = order["price"]
    adet = order["origQty"]

    sorgu_metni = f'''
    INSERT INTO satis_plani (
                                   k_id,
                                   symbol,
                                   orderId,
                                   adet,
                                   fiyat
                               )
                               VALUES (
                                   '{k_id}',
                                   '{symbol}',
                                   '{orderId}',
                                   '{adet}',
                                   '{fiyat}'
                               );
    '''
    return Komut(sorgu_metni)

def Satis_Taslak_Plandaki_Emirleri_Yolla(k_id , satis_plani):
    kunye = PandasSQL(f"SELECT * FROM kunye WHERE k_id = {k_id};").loc[0]
    sembol = kunye["sembol"]

    def Limit_Emri_Yolla_Kontrolcu(x):
        durum , order = Limit_Emri_Yolla(sembol, x["adet"] , x["fiyat"] , "SELL")
        x["durum"] = durum
        x["order"] = order
        if durum:
            Kapanmamis_Emirlere_Kayit_Ekle(k_id , order)
            Satis_Planina_Kayit_Ekle(k_id , order)
        return x
            
    satis_plani = satis_plani.apply(lambda x: Limit_Emri_Yolla_Kontrolcu(x) , axis=1)
    return satis_plani

In [ ]:
#@title Emir Yollamak İle İlgili Fonksiyonlar

def Tutar_Ile_Market_Emri_Al(k_id ,symbol, tutar):

    # Borsadan alınmış son fiyata bakalım.
    tickers = PandasSQL("Select * from tickers")
    mevcut_fiyat = tickers[tickers["symbol"] == "BTCUSDT"]["price"].values[0]
    mevcut_fiyat = float(mevcut_fiyat)
    # Borsadan alınmış son fiyata bakalım.

    '''
    Tutar ile market emri almaya yardımcı olacak fonksiyondur.

    Kontroller
    1. Tutar Min_Notion'dan büyük olmalı.
    2. Bu tutar ile güncel fiyattan alınacak adetteki coin miktarı minQTY'den büyük olmalı.
    3. Tutar ile market emri aldığımız için yardımcı para biriminin küsürat sayısı kadarlık bir küsürat koymamız lazım.
    '''
    
    # aşağıdaki kod ile tutarın sadece virgülden sonraki kaç hassassiyeti varsa o kadarlık kısmını alacağız.
    sorgu_metni = f'''
        SELECT quoteAssetPrecision
        FROM exchange_info
        WHERE symbol = '{symbol}'
    '''
    hassasiyet = int(Tek_Hucre(sorgu_metni))
    tutar = math.floor(tutar*pow(10,hassasiyet))/pow(10,hassasiyet)
    # aşağıdaki kod ile tutarın sadece virgülden sonraki kaç hassassiyeti varsa o kadarlık kısmını alacağız.

    # Burada kontrolleri yapalım.

    sorgu_metni = f'''
    SELECT *
    FROM exchange_info
    WHERE symbol = '{symbol}';
    '''

    kurallar = PandasSQL(sorgu_metni).head(1)
    filtreler = kurallar["filters"].values[0]
    list_data = ast.literal_eval(filtreler)
    fixed_data = json.dumps(list_data)
    dict_data = json.loads(fixed_data)

    # minNotional miktarını çekelim. 
    price_filter_items = [item for item in dict_data if item['filterType'] == 'MIN_NOTIONAL']
    minNotional = float(price_filter_items[0]['minNotional'])

    # minQty miktarını çekelim.
    miktar_filtreleri = [item for item in dict_data if item['filterType'] == 'LOT_SIZE']
    minQty = float(miktar_filtreleri[0]['minQty'])

    # Burada kontrolleri yapalım.
    if tutar < minNotional:
        return "minNotional kuralı sağlanmıyor." , 0
    if tutar/mevcut_fiyat <minQty:
        return "minQty kuralı sağlanmıyor." , 0

    try:
        order = client.create_order(
        symbol=symbol,
        side='BUY',
        type='MARKET',
        quoteOrderQty=tutar)

        if order["status"] == "FILLED":
            orderId = order["orderId"]
            Trade_Kasa_Etkisi(k_id,symbol,orderId)
            return "FILLED" , order["orderId"]

        elif order["status"] == "PARTIALLY_FILLED":
            orderId = order["orderId"]
            return "PARTIALLY_FILLED" , order["orderId"]
        else:
            return "Emir yollandı ama bilinmeyen bir hata aldik." , order["orderId"]
    except Exception as e:
        return "Emri yollayamadık." ,str(e)

def Limit_Emri_Yolla(symbol, quantity , price , side):
    try:
        order = client.create_order(
        symbol=symbol,
        side=side,
        type="LIMIT",
        timeInForce="GTC",
        quantity=quantity,
        price=price)

        orderId = order["orderId"]

        if isinstance(orderId, int):
            return True , order
        else:
            return False , str(order)
    
    except Exception as e:
        return False , str(e)

Kalibrasyon Fonksiyonu Tek Tek Kodlayalım

In [ ]:
k_id = 1

In [ ]:
# Künyedeki kayıtlı emirleri temizleyelim.
durum , kapanmamis_emirler = Kunyedeki_Emirleri_Temizle(k_id)

if durum:
    if len(kapanmamis_emirler.index) > 0:
        print("Kapanmamis emirler basariyle kapatildi.")
        display(kapanmamis_emirler)
    else:
        print("Kapatilacak emir yok.")
else:
    print("Emirler basariyle kapatilamadi.")
    display(kapanmamis_emirler)
    print("Buraya return komutu yazılacak.")

Kapatilacak emir yok.


In [ ]:
# Eldeki tüm nakit ile piyasa emrinden p1 satın alalım. 

# Eldeki kullanılabilir nakidi bulalım.
kunye = PandasSQL(f"SELECT * FROM kunye WHERE k_id = {k_id};").loc[0]

p2_miktari = float(kunye["p2_miktari"])
p2_miktari
# Eldeki kullanılabilir nakidi bulalım.

-89.99012382000001

In [ ]:
# Bu nakitle market emrini kullanarak p1 alalım.
cevap , metin = Tutar_Ile_Market_Emri_Al(k_id , kunye["sembol"] , p2_miktari )

In [ ]:
if cevap == "FILLED":
    print("Islem Basarili")
elif cevap == "PARTIALLY_FILLED":
    print("Burada biseler düsünmemiz lazım.")
elif cevap == "minNotional kuralı sağlanmıyor." or cevap == "minQty kuralı sağlanmıyor.":
    print("Elde az p2 olduğu için yeni p2 alamıyoruz. Sorun yok.")
elif cevap == "Emir yollandı ama bilinmeyen bir hata aldik.":
    print("Burada metin adlı değişkeni telegrama yollayarak sistemi durduracağız.")
else:
    print("Cevap ve metin telegramdan yollanarak sistem durdurulacak.")

Islem Basarili


In [ ]:
# Eldeki tüm p1'ler ile satış planı oluşturalım.
durum , satis_plani  = Satis_Plani_Olusturucu(k_id)

#if not durum:
 #   return durum , satis_plani
# Eldeki tüm p1'ler ile satış planı oluşturalım.

In [ ]:
# Satis planini borsaya yollayalim.
sonuc_tablosu = Satis_Taslak_Plandaki_Emirleri_Yolla(1 , satis_plani)
# Aşağıdaki kodu yazmamın sebebi durum değeri true false yerine 1-0 olarak geliyordu.
if int(sonuc_tablosu["durum"].sum()) == int(len(sonuc_tablosu.index)):
    print("Basarili")
else:
    print("Hata")
    print(sonuc_tablosu)

Basarili
